## Propose rules for one completed SOURCE run

Run the notebook from the top to create a new proposal batch. Review individual proposals after the batch is stored.

In [ ]:
dbutils.widgets.text("source_run_id", "", "Completed SOURCE run ID")
dbutils.widgets.dropdown("balance_pilot", "OFF", ["OFF", "ON"], "Balance pilot")

In [0]:
proposals_table = "workspace.synthetic_profile.ai_rule_proposals"
batches_table = "workspace.synthetic_profile.ai_rule_batches"

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.synthetic_profile.ai_rule_batches (
    batch_id STRING,
    source_run_id STRING,
    created_at TIMESTAMP,
    scope_name STRING,
    model_endpoint STRING,
    prompt_sha256 STRING,
    response STRING
) USING DELTA
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.synthetic_profile.ai_rule_proposals (

    proposal_id STRING,
    created_at TIMESTAMP,
    source_run_id STRING,

    target_table STRING,
    target_column STRING,

    rule_type STRING,

    parent_columns ARRAY<STRING>,

    confidence STRING,
    evidence_type STRING,

    statistical_evidence STRING,
    rationale STRING,
    proposed_rule STRING,

    requires_sme_review BOOLEAN,
    approval_status STRING,
    batch_id STRING,
    reviewed_by STRING,
    reviewed_at TIMESTAMP,
    legacy_approval_status STRING

)
USING DELTA
""")

# Migrate the original table in place. Rows without a batch predate verifiable provenance.
provenance_columns = {
    "batch_id": "STRING",
    "reviewed_by": "STRING",
    "reviewed_at": "TIMESTAMP",
    "legacy_approval_status": "STRING",
}
existing_columns = set(spark.table(proposals_table).columns)
missing_columns = [
    f"{name} {data_type}"
    for name, data_type in provenance_columns.items()
    if name not in existing_columns
]
if missing_columns:
    spark.sql(f"ALTER TABLE {proposals_table} ADD COLUMNS ({', '.join(missing_columns)})")

spark.sql("""
UPDATE workspace.synthetic_profile.ai_rule_proposals
SET legacy_approval_status = 'APPROVED', approval_status = 'PENDING'
WHERE batch_id IS NULL AND approval_status = 'APPROVED'
""")

In [0]:
from uuid import uuid4

from pyspark.sql import functions as F

requested_source_run_id = dbutils.widgets.get("source_run_id").strip()
balance_pilot = dbutils.widgets.get("balance_pilot") == "ON"
if balance_pilot and not requested_source_run_id:
    raise ValueError("Set source_run_id before creating a balance pilot proposal batch.")

scope_name = "BALANCE_PILOT" if balance_pilot else "ACCOUNT_CORE"

source_run = spark.sql("""
    SELECT run_id
    FROM workspace.synthetic_profile.profile_runs
    WHERE profile_type = 'SOURCE'
      AND status = 'COMPLETED'
      AND completed_at IS NOT NULL
      AND run_id IS NOT NULL
      AND (:requested_source = '' OR run_id = :requested_source)
    ORDER BY completed_at DESC, started_at DESC, run_id DESC
    LIMIT 1
""", args={"requested_source": requested_source_run_id}).first()

if source_run is None:
    if requested_source_run_id:
        raise ValueError(f"Run {requested_source_run_id} is unavailable or is not a completed SOURCE profile.")
    raise RuntimeError("Complete a SOURCE profile in Notebook 01 before proposing rules.")

source_run_id = source_run["run_id"]
batch_id = str(uuid4())
spark.createDataFrame(
    [(source_run_id, batch_id, scope_name)],
    "source_run_id STRING, batch_id STRING, scope_name STRING",
).createOrReplaceTempView("ai_batch_context")

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW ai_column_context
AS

WITH category_ranked AS (

    SELECT
        c.logical_table_name,
        c.column_name,
        c.category_value,
        c.proportion,

        ROW_NUMBER() OVER (
            PARTITION BY
                c.logical_table_name,
                c.column_name
            ORDER BY c.proportion DESC, c.category_value
        ) AS rn

    FROM workspace.synthetic_profile.category_profile c

    JOIN ai_batch_context b
        ON c.run_id = b.source_run_id
),

categories AS (

    SELECT
        logical_table_name,
        column_name,

        COLLECT_LIST(
            NAMED_STRUCT(
                'value', category_value,
                'proportion', proportion
            )
        ) AS top_categories

    FROM category_ranked

    WHERE rn <= 10

    GROUP BY
        logical_table_name,
        column_name
)

SELECT
    b.source_run_id,
    b.batch_id,
    b.scope_name,
    q.logical_table_name,
    q.column_name,
    q.data_type,
    CASE
        WHEN upper(q.column_name) IN (
            'PARTY_ID', 'FROM_PARTY_ID', 'TO_PARTY_ID', 'ACCT_ID', 'ACCOUNT_ID',
            'ACCOUNT_NUMBER', 'REQ-CIF-KEY', 'REQ-CIF-TIE-BREAKER'
        ) THEN 'IDENTIFIER'
        WHEN upper(q.data_type) IN (
            'BYTE', 'SHORT', 'INT', 'INTEGER', 'LONG', 'BIGINT',
            'FLOAT', 'DOUBLE', 'DECIMAL'
        ) THEN 'NUMERIC'
        WHEN upper(q.data_type) = 'DATE' THEN 'DATE'
        WHEN upper(q.data_type) IN ('TIMESTAMP', 'TIMESTAMP_NTZ') THEN 'TIMESTAMP'
        WHEN upper(q.data_type) IN ('STRING', 'VARCHAR', 'CHAR') THEN 'CATEGORICAL_CANDIDATE'
        ELSE 'OTHER'
    END AS column_role,

    q.null_pct,
    q.distinct_count,

    n.mean_value,
    n.p25,
    n.p50,
    n.p75,
    n.p95,
    n.stddev_value,

    c.top_categories

FROM workspace.synthetic_profile.column_quality q

JOIN ai_batch_context b
    ON q.run_id = b.source_run_id

LEFT JOIN workspace.synthetic_profile.numeric_profile n
    ON n.run_id = b.source_run_id
   AND n.logical_table_name = q.logical_table_name
   AND n.column_name = q.column_name

LEFT JOIN categories c
    ON c.logical_table_name = q.logical_table_name
   AND c.column_name = q.column_name

WHERE q.logical_table_name IN (
    'TKQ_ACCOUNT',
    'TKQ_ACCOUNT_BALANCE'
);

Preview the aggregate column context used to build this batch's prompt.

In [0]:
%sql

SELECT *
FROM ai_column_context
ORDER BY logical_table_name, column_name;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW ai_account_rule_input AS

SELECT
    source_run_id,
    batch_id,
    scope_name,

    CONCAT_WS(
        '\n',
        COLLECT_LIST(
            TO_JSON(
                NAMED_STRUCT(
                    'table', logical_table_name,
                    'column', column_name,
                    'type', data_type,
                    'role', column_role,

                    'null_pct', null_pct,
                    'distinct_count', distinct_count,

                    'mean', mean_value,
                    'p25', p25,
                    'p50', p50,
                    'p75', p75,
                    'p95', p95,

                    'top_categories', top_categories
                )
            )
        )
    ) AS profile_summary

FROM ai_column_context
GROUP BY source_run_id, batch_id, scope_name
HAVING COUNT(DISTINCT logical_table_name) = 2;

In [0]:
request = spark.table("ai_account_rule_input").first()
if request is None or not request["profile_summary"]:
    raise RuntimeError("The selected SOURCE run needs aggregate column profiles for both account tables.")

pilot_instructions = ""
if request["scope_name"] == "BALANCE_PILOT":
    pilot_instructions = (
        "\n\nBalance pilot scope:\n"
        "Propose at most one DERIVED rule for each of AVAILABLE_TO_BUY_AMT and OVERLIMIT_AMT "
        "on TKQ_ACCOUNT_BALANCE. The only eligible parents are TKQ_ACCOUNT.CREDIT_LIMIT_AMT "
        "and TKQ_ACCOUNT_BALANCE.CURRENT_BALANCE_AMT. The existing formulas are "
        "max(credit limit - current balance, 0) and max(current balance - credit limit, 0), respectively. "
        "Assess these existing dependencies and omit unsupported proposals. "
        "Keep domain inference distinct from statistical evidence and leave other dependencies out."
    )

prompt = """You are assisting with a banking synthetic-data generation system.

You receive ONLY aggregate statistical profiles. You do not have row-level records.

Your task is to propose candidate generation dependencies and business rules.

A good candidate rule must:
1. improve logical or statistical consistency;
2. identify fields that should not be generated independently;
3. distinguish statistical evidence from domain inference;
4. avoid inventing relationships unsupported by the supplied evidence;
5. mark domain-inferred rules for SME review;
6. prefer deterministic DERIVED rules where a field is clearly derived from others;
7. use CONDITIONAL when a target distribution should depend on another variable;
8. use CORRELATED only when a numeric association should be preserved but no deterministic formula is supported;
9. use NO_RULE when independent generation is acceptable or evidence is insufficient.

Do NOT claim causation from correlation.

For each proposed rule identify:
- target table
- target column
- rule type
- parent columns
- confidence: HIGH, MEDIUM, or LOW
- evidence type: STATISTICAL, DOMAIN_INFERENCE, or STATISTICAL_AND_DOMAIN
- statistical evidence
- rationale
- proposed generation rule
- whether SME review is required""" + pilot_instructions + "\n\nHere are the profiler statistics:\n\n" + request["profile_summary"]
spark.createDataFrame(
    [(request["source_run_id"], request["batch_id"], request["scope_name"], prompt)],
    "source_run_id STRING, batch_id STRING, scope_name STRING, prompt STRING",
).createOrReplaceTempView("ai_model_request")

In [0]:
%sql

SELECT source_run_id, batch_id, scope_name, sha2(prompt, 256) AS prompt_sha256
FROM ai_model_request;

In [0]:
if spark.table(batches_table).filter(F.col("batch_id") == request["batch_id"]).first() is None:
    spark.sql("""
    INSERT INTO workspace.synthetic_profile.ai_rule_batches
        (batch_id, source_run_id, created_at, scope_name, model_endpoint, prompt_sha256, response)
    SELECT
        batch_id,
        source_run_id,
        current_timestamp(),
        scope_name,
        'system.ai.gpt-oss-20b',
        sha2(prompt, 256),
        ai_query(
            'system.ai.gpt-oss-20b',
            prompt,
            responseFormat => '{
          "type": "json_schema",
          "json_schema": {
            "name": "banking_generation_rules",
            "schema": {
              "type": "object",
              "properties": {
                "rules": {
                  "type": "array",
                  "items": {
                    "type": "object",
                    "properties": {
                      "target_table": {
                        "type": "string"
                      },
                      "target_column": {
                        "type": "string"
                      },
                      "rule_type": {
                        "type": "string"
                      },
                      "parent_columns": {
                        "type": "array",
                        "items": {
                          "type": "string"
                        }
                      },
                      "confidence": {
                        "type": "string"
                      },
                      "evidence_type": {
                        "type": "string"
                      },
                      "statistical_evidence": {
                        "type": "string"
                      },
                      "rationale": {
                        "type": "string"
                      },
                      "proposed_rule": {
                        "type": "string"
                      },
                      "requires_sme_review": {
                        "type": "boolean"
                      }
                    },
                    "required": [
                      "target_table",
                      "target_column",
                      "rule_type",
                      "parent_columns",
                      "confidence",
                      "evidence_type",
                      "statistical_evidence",
                      "rationale",
                      "proposed_rule",
                      "requires_sme_review"
                    ],
                    "additionalProperties": false
                  }
                }
              },
              "required": [
                "rules"
              ],
              "additionalProperties": false
            },
            "strict": true
          }
        }'
        )
    FROM ai_model_request
    """)
else:
    print(f"Reusing the stored response for batch {request['batch_id']}. Run from the top to start a new batch.")

In [0]:
%sql

SELECT b.batch_id, b.source_run_id, b.scope_name, b.created_at, b.model_endpoint, b.prompt_sha256, b.response
FROM workspace.synthetic_profile.ai_rule_batches b
JOIN ai_model_request r
    ON b.batch_id = r.batch_id AND b.source_run_id = r.source_run_id;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW ai_parsed_rule_proposals AS

SELECT
    b.batch_id,
    b.source_run_id,

    FROM_JSON(
        b.response,

        'STRUCT<
            rules:ARRAY<
                STRUCT<
                    target_table:STRING,
                    target_column:STRING,
                    rule_type:STRING,
                    parent_columns:ARRAY<STRING>,
                    confidence:STRING,
                    evidence_type:STRING,
                    statistical_evidence:STRING,
                    rationale:STRING,
                    proposed_rule:STRING,
                    requires_sme_review:BOOLEAN
                >
            >
        >'
    ) AS parsed_response

FROM workspace.synthetic_profile.ai_rule_batches b
JOIN ai_model_request r
    ON b.batch_id = r.batch_id AND b.source_run_id = r.source_run_id;

In [0]:
parsed = spark.table("ai_parsed_rule_proposals").selectExpr(
    "parsed_response.rules AS rules"
).first()
if parsed is None or parsed["rules"] is None:
    raise RuntimeError("The stored batch did not return a valid rules array. Inspect its saved response.")

spark.sql("""
INSERT INTO workspace.synthetic_profile.ai_rule_proposals (
    proposal_id, created_at, source_run_id, target_table, target_column,
    rule_type, parent_columns, confidence, evidence_type, statistical_evidence,
    rationale, proposed_rule, requires_sme_review, approval_status, batch_id
)
SELECT
    uuid(),
    current_timestamp(),
    p.source_run_id,
    rule.target_table,
    rule.target_column,
    rule.rule_type,
    rule.parent_columns,
    rule.confidence,
    rule.evidence_type,
    rule.statistical_evidence,
    rule.rationale,
    rule.proposed_rule,
    rule.requires_sme_review,
    'PENDING',
    p.batch_id
FROM (
    SELECT parsed.*
    FROM ai_parsed_rule_proposals parsed
    LEFT ANTI JOIN workspace.synthetic_profile.ai_rule_proposals stored
        ON parsed.batch_id = stored.batch_id
) p
LATERAL VIEW EXPLODE(p.parsed_response.rules) exploded AS rule
""")

In [0]:
%sql

SELECT
    p.source_run_id,
    p.batch_id,
    proposal_id,
    target_table,
    target_column,
    rule_type,
    parent_columns,
    confidence,
    evidence_type,
    statistical_evidence,
    proposed_rule,
    rationale,
    requires_sme_review,
    approval_status,
    reviewed_by,
    reviewed_at

FROM workspace.synthetic_profile.ai_rule_proposals p
JOIN ai_model_request r
    ON p.batch_id = r.batch_id AND p.source_run_id = r.source_run_id

ORDER BY
    target_table,
    target_column;

### Review one proposal

Inspect the proposal and its batch before making a decision. Complete any required SME review. To record a decision, run the following statement separately in the Databricks SQL editor. Supply the exact SOURCE, batch, and proposal IDs and choose APPROVED or REJECTED.

```sql
UPDATE workspace.synthetic_profile.ai_rule_proposals
SET approval_status = :review_decision,
    reviewed_by = current_user(),
    reviewed_at = current_timestamp()
WHERE source_run_id = :source_run_id
  AND batch_id = :batch_id
  AND proposal_id = :proposal_id
  AND approval_status = 'PENDING'
  AND reviewed_at IS NULL
  AND :review_decision IN ('APPROVED', 'REJECTED');
```

Approval records a proposal review. Implementation and execution are separate decisions. Legacy rows have no batch ID and are ineligible for review through this workflow. Re-propose them in a new batch if needed.

Provenance: completed SOURCE run → proposal batch → proposal ID → review status, reviewer, and review time.